# Clamped beam with the common MORFE API

This conservative St. Venant–Kirchhoff example uses only the public, physics-independent MORFE workflow. The committed output uses order 3. To reproduce the order-9 reference, change `order = 3` to `order = 9` and rerun the notebook.

Before the first run, initialise the example environment from the MORFEFerrite repository root with `julia --project=examples/01_clamped_beam_ferrite -e 'using Pkg; Pkg.develop(path="."); Pkg.instantiate()'`. This selects MORFEFerrite from the current checkout and downloads MORFE from Julia's package registry.

The first cell activates and instantiates that environment. `SVK` is only a short name for the structural backend used to describe the mechanical case; `build_model` and `parametrise` belong to the common API.

In [1]:
import Pkg
Pkg.activate(@__DIR__; io = devnull)
Pkg.instantiate(; io = devnull)
using MORFE, MORFEFerrite
const SVK = MORFEFerrite.StructuralSVK # for Saint-Venant-Kirchhoff hyperelasticity

MORFEFerrite.StructuralSVK

## 1. Describe the mechanical case

Choose the polynomial expansion order directly: `3` is the quick demonstration and `9` is the conservative reference calculation.

`mechanical_model` reads the mesh and supplies the physical information required by the structural backend. Here the Rayleigh damping coefficients are zero, so the beam is conservative. `dirichlet = "Dirichlet"` names a physical facet group stored in the Gmsh mesh; every displacement component on those labeled facets is fixed to zero, producing the clamped ends. The finite-element and quadrature orders use their API defaults.

In [2]:
order = 3  # Change to 9 for the reference calculation.
case = SVK.mechanical_model(joinpath(@__DIR__, "clamped_clamped_beam.msh");
    material = SVK.SVKMaterial(E = 160e3, ν = 0.22, ρ = 2.32e-3),
    damping = SVK.RayleighDamping(α = 0.0, β = 0.0),
    dirichlet = "Dirichlet")

Info    : Reading '/Users/tiago/Desktop/AM-TUM/Code/MORFEFerrite/MORFEFerrite.jl/examples/01_clamped_beam_ferrite/clamped_clamped_beam.msh'...
Info    : 27 entities
Info    : 328 nodes
Info    : 270 elements
Info    : Done reading '/Users/tiago/Desktop/AM-TUM/Code/MORFEFerrite/MORFEFerrite.jl/examples/01_clamped_beam_ferrite/clamped_clamped_beam.msh'


AssembledMechanicalModel (Ferrite)
  free DOFs : 4977 (of 5103)
  material  : SVK  E=160000.0  ν=0.22  ρ=0.00232
  damping   : Rayleigh  α=0.0  β=0.0

## 2. Build the MORFE model

`build_model` converts the structural case into MORFE's physics-independent model and computes its spectral data. `master = [1]` selects the first vibration-mode pair as the tangent space of the reduced model. The returned `meta` contains auxiliary backend information, including the selected eigenvalues displayed below.

In [3]:
(; model, spectral, meta) = build_model(case; master = [1], expansion_order = order)
meta.spectrum.eigenvalues[meta.master_indices] # print master eigenvalues

2-element Vector{ComplexF64}:
  0.0 + 0.5374120526400513im
 -0.0 - 0.5374120526400513im

## 3. Parametrise the invariant manifold

`parametrise` computes the polynomial manifold map `W` and its reduced dynamics `R` up to the chosen order. The resonance configuration keeps near-resonant monomials in complex normal form. Evaluating `R` at the end of the cell displays the reduced system.

In [4]:
W, R = parametrise(model, spectral, order;
    resonance = ResonanceConfig(style = :complex_normal_form, tol = 0.05))
R # print reduced dynamics

┌ Info: conjugate_permutation is active — the following assumptions must hold:
│   1. Real-valued FOM: all matrices in model.linear_terms and all nonlinear/force terms must have real-valued entries (eltype <: Real or purely-real complex).
│   2. Each mode either comes in a complex conjugate pair with another mode, or is self-paired  meaning it has a real eigenvalue and a real-valued mode shape.
│   3. Eigenvalue conjugacy is necessary but NOT sufficient for paired modes; the right eigenvectors must satisfy master_right_modes[:, perm[r]] = conj(master_right_modes[:, r]).
│   4. If external modes are present (N_EXT > 0): the same pairing rules apply to the external eigenvalues, encoded in the NVAR-length permutation.
└ Passing an incorrect permutation silently corrupts the parametrisation and reduced-dynamics.
┌ Info: Using optimised FEM path: cached per-element loop
└   n_fem_terms = 2


ReducedDynamics{2, 2, ComplexF64}(DensePolynomial{ComplexF64, 2, 2, Matrix{ComplexF64}}(ComplexF64[0.0 + 0.5374120526400513im 0.0 + 0.0im … 0.0 - 0.0im 0.0 - 0.0im; 0.0 + 0.0im -0.0 - 0.5374120526400513im … -8.321010769782508e-19 - 2.4475346813030787e-5im 0.0 - 0.0im], MultiindexSet{2}(StaticArraysCore.SVector{2, Int64}[[1, 0], [0, 1], [2, 0], [1, 1], [0, 2], [3, 0], [2, 1], [1, 2], [0, 3]], [0, 0, 2, 5, 9]), [3, 3]), 0)

## Optional: save the standard ROM files

The common saver writes `W`, `R`, their coefficient table, and a summary to the example's `results` directory. This cell is optional; the ROM is already available in memory after the previous cell.

In [5]:
MORFE.save_rom(joinpath(@__DIR__, "results"), W, R);